# FuseChain: Data Fusion

This notebook merges the **Balanced On-Chain Data** with the **Selected Off-Chain Features** to create the final training dataset for XGBoost.

## Inputs
- **On-Chain**: `balanced_label_days.parquet` (Address-Level, Balanced)
- **Off-Chain**: `offchain_daily_features.parquet` (Global Daily, Top 6 Features)

## Output
- `final_train_data.parquet`

## Final Feature Set (11 Features)
1. normal_total_cnt
2. active_span_min
3. uniq_peers_cnt
4. burst_max_tx_5m
5. normal_sent_cnt
6. `reddit_fraud_ratio`
7. `reddit_sentiment_deviation`
8. `reddit_total_activity`
9. reddit_avg_sentiment
9. eth_volatility_7d
10. eth_volume_change_pct
11. eth_daily_return
12. eth_intraday_volatility


## Selected Features

### Market Features
- **mkt_volatility_7d**: 7-day rolling volatility (market stress)
- **mkt_volume_change_pct**: % change in volume
- **mkt_daily_return**: Significant price crash/gain

### Reddit (Social) Features
- **reddit_fraud_ratio**: % of scam-related comments
- **reddit_sentiment_deviation**: Mood drop compared to baseline
- **reddit_total_activity**: General community buzz

In [ ]:
import pandas as pd
import os

# Paths
onchain_path = '../data/processed/chain/balanced_label_days.parquet'
offchain_path = '../data/processed/offchain_daily_features.parquet'
output_dir = '../data/processed/final'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Load Data
print("Loading On-Chain Data...")
chain_df = pd.read_parquet(onchain_path)
print(f"On-Chain Shape: {chain_df.shape}")

print("\nLoading Off-Chain Data...")
off_df = pd.read_parquet(offchain_path)
print(f"Off-Chain Shape: {off_df.shape}")

# Ensure date format matches
chain_df['day'] = pd.to_datetime(chain_df['day']).dt.date
off_df['day'] = pd.to_datetime(off_df['day']).dt.date

In [ ]:
# Merge: Left Join (Keep all labeled on-chain days)
merged_df = pd.merge(chain_df, off_df, on='day', how='left')

# Fill missing off-chain data with 0 (for days with no active market/reddit data)
merged_df = merged_df.fillna(0)

print(f"\nMerged Shape: {merged_df.shape}")

In [ ]:
# === FINAL FEATURE SELECTION ===
final_features = [
    # Identity & Target
    'address', 'day', 'is_anomalous',

    # On-Chain (Top 5)
    'eth_net_flow',
    'burst_max_tx_5m',
    'uniq_peers_cnt',
    'eth_sent_sum',
    'active_span_min',

    # Reddit (Top 3)
    'reddit_fraud_ratio',
    'reddit_sentiment_deviation',
    'reddit_total_activity',

    # Market (Top 3)
    'mkt_volatility_7d',
    'mkt_volume_spike',
    'mkt_daily_return'
]

# Keep only selected features
train_df = merged_df[final_features].copy()

print(f"Final Training Set Shape: {train_df.shape}")
print("\nClass Distribution:")
print(train_df['is_anomalous'].value_counts())

train_df.head()

In [ ]:
# Save
output_file = os.path.join(output_dir, 'final_train_data.parquet')
train_df.to_parquet(output_file, index=False)
print(f"\n✅ Saved Final Training Data to: {output_file}")